## Initialization

In [ ]:
# Importing needed code

import re
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.stats import linregress

from data_processing.arc_paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    find_failed_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.helpers import (
    stop, get_input_with_default, input_experiment_ids, get_midpoints_from_min_max_series)
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
def load_rate_data(experiment_name: str, time_bin_length: int) -> pd.DataFrame:
    file_name = f"{experiment_name}_data_{time_bin_length}s_bin.csv"
    output_root = get_report_root(experiment_name)
    file_path = output_root / file_name
    df = pd.read_csv(file_path)
    return df

In [ ]:
StableInterval = tuple[float, float]  # in ps after exp. start
StableIntervals = tuple[StableInterval, StableInterval, StableInterval]


def find_stable_region(
    rate_df: pd.DataFrame,
    duration: float | None = None
) -> StableInterval:
    # TODO let user enter stable regions
    # check stability via linear fit
    # show slope to user
    # let user confirm stable region, or re-enter region
    done = False
    start = None
    end = None
    while not done:
        while start is None:
            in_val = input("Enter start time (in minutes after experiment start): ")
            try:
                start = float(in_val)
            except ValueError:
                print("Not a valid number, try again")
        if duration is not None:
            end = start + duration
        while end is None:
            in_val = input("Enter start time (in minutes after experiment start): ")
            try:
                end = float(in_val)
            except ValueError:
                print("Not a valid number, try again")
        if end <= start:
            print("End must be greater than start, try again")
            start = None
            end = None
            continue

        x_all = rate_df["Bin time (s)"]
        y_all = rate_df["Neutron rate (cps)"]
        duration_series_idx = x_all.between(start * 60, end * 60)
        x = x_all[duration_series_idx]
        y = y_all[duration_series_idx]
        result = linregress(x, y)
        print(f"Rate slope = {result.slope:.2E} cps/s +/- {result.stderr:.2E}")
        in_done = get_input_with_default(
            """\
Is this region stable?
Enter y/n, or press Enter for no
""",
            "n",
            str
        )
        done = in_done != "n"
    return start, end


def find_stable_regions(rate_df: pd.DataFrame) -> StableIntervals:
    bg_region = find_stable_region(rate_df)
    bg_start, bg_end = bg_region
    bg_duration = bg_end - bg_start

    beam_region = find_stable_region(rate_df, bg_duration)
    ecell_region = find_stable_region(rate_df, bg_duration)
    return bg_region, beam_region, ecell_region
    # done = False
    # while not done:
    #     stable_regions = (
    #         find_stable_region(rate_df),
    #         find_stable_region(rate_df),
    #         find_stable_region(rate_df)
    #     )
    #     durations = [end-start for start, end in stable_regions]
    #     equality_check = [duration == durations[0] for duration in durations]
    #     print(durations)
    #     print(equality_check)
    #     are_equal = all(equality_check)
    #     done = are_equal
    #     if not are_equal:
    #         print("All values should be equal, try again")
    # return stable_regions

## Input Settings

In [ ]:
experiment_ids = input_experiment_ids()

In [ ]:
time_bin_length = get_input_with_default(
    """\
Enter bin length (in seconds)
Press Enter for default (30)
""",
    30,
    int
)

## Data Loading

In [ ]:
all_experiment_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# TODO load unclassified data
# TODO load 30s binned data
for exp_id, exp_data in all_experiment_data.items():
    detector_df = load_psd(exp_id)
    rate_df = load_rate_data(exp_id, time_bin_length)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = detector_df
    exp_data[ExperimentDataKey.ALL_BINNED_DATA] = rate_df

In [ ]:
# rate_df = all_experiment_data["ID-423"][ExperimentDataKey.ALL_BINNED_DATA]
# find_stable_regions(rate_df)

## Analysis

In [ ]:
# TODO show rate graph
for exp_id, exp_data in all_experiment_data.items():
    binned_neutrons = exp_data[ExperimentDataKey.ALL_BINNED_DATA]

    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    rate_errors = binned_neutrons[n_error_col_name]

    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
    dot_size = 8
    ax.errorbar(
        zeroed_bins,
        rates,
        yerr=rate_errors,
        fmt=".",
        linestyle='',
        markersize=dot_size,
        capsize=dot_size
    )
    ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("Neutron count rate [1/s]", fontsize=14)
    ax.tick_params(labelsize=12)
    # ax.set_ylim(0, 22)
    # ax.set_ylim(3.5, 5.0)
    # _, x_end = ax.get_xlim()
    # ax.xaxis.set_ticks(np.arange(0, x_end, 5))
    ax.xaxis.set_minor_locator(plt.MultipleLocator(5))

    # Add a title above the plot
    fig.text(
        0.5,
        0.90,
        f"{exp_id} neutron count rate over time (Dwell time {time_bin_length}s)",
        ha='center',
        fontsize=20
    )
    ax.grid(which="both")

    # Show the plot (optional)
    plt.show()

In [ ]:
# TODO let user enter stable regions
# check stability via linear fit
# show slope to user
# let user confirm stable region, or re-enter region
for exp_id, exp_data in all_experiment_data.items():
    event_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    rate_df = exp_data[ExperimentDataKey.ALL_BINNED_DATA]
    event_counts = []

    stable_regions = find_stable_regions(rate_df)
    print(f"-----Experiment {exp_id}-----")
    print("---Stable Regions---")
    for i, region in enumerate(stable_regions):
        start, end = region
        print(f"Region {i+1}: {start:.1f} min to {end:.1f} min")
        print(f"Duration: {(end-start):.1f} min")

        time_col = event_df['TIMETAG']
        # count = event_df[event_df[
        #     (start * 1e12) < time_col & time_col <= (end * 1e12)
        # ]].shape[0]
        is_in_region = time_col.between(start * 60 * 1e12, end * 60 * 1e12)
        events_in_region = event_df[
            is_in_region
        ]
        event_count = events_in_region.shape[0]
        event_counts.append(event_count)

    bg_count, beam_count, ecell_count = event_counts
    print("---Counts---")
    print(f"Background:   {bg_count}")
    print(f"Beam Loading: {beam_count}")
    print(f"Ecell:        {ecell_count}")

    beam_diff = beam_count - bg_count
    ecell_diff = ecell_count - bg_count
    print("---Relative to Background---")
    print(f"Beam Loading: {beam_diff}")
    print(f"Ecell:        {ecell_diff}")

    ecell_relative_increase = ((ecell_diff - beam_diff) / beam_diff) * 100
    print()
    print(f"Percent Increase: {ecell_relative_increase:.2f}%")